# Lab 1: Monthly Data-Quality Monitoring with TFDV and ML Metadata

In the original walkthrough you used ML Metadata (MLMD) to record a single TFDV step: one dataset in, one inferred schema out. Then you walked back from the schema to the dataset.

In production, schema inference happens once. What runs every day or month is **validation**: new data arrives, gets checked against the schema, and someone decides whether to accept it, reject it, or change the schema. Months later, someone will ask *why* a batch was rejected, or *why* the schema changed. MLMD is where those answers live.

In this lab you will build that monitoring loop by hand with the MLMD API and TFDV:

1. Infer and curate a baseline schema (version 0 and version 1) from the first quarter of 2011.
2. Validate six monthly batches against the schema and against the previous accepted month (drift).
3. Apply a decision policy to each batch: **ACCEPTED**, **ACCEPTED_WITH_DRIFT**, **REJECTED**, or **NEEDS_SCHEMA_UPDATE**.
4. Evolve the schema when the data legitimately changes, and record *which anomaly report triggered each change*.
5. Validate a serving batch (no labels) using schema environments.
6. Answer audit questions using only the metadata store.

**Dataset:** [UCI Bike Sharing](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) hourly rentals in Washington D.C., 2011-2012 (CC BY 4.0).

## Imports

In [3]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import os
import hashlib
import urllib.request
import zipfile

import numpy as np
import pandas as pd

import tensorflow as tf
import tensorflow_data_validation as tfdv
from tensorflow_metadata.proto.v0 import schema_pb2, anomalies_pb2

from ml_metadata.metadata_store import metadata_store
from ml_metadata.proto import metadata_store_pb2

print('TF version:', tf.__version__)
print('TFDV version:', tfdv.version.__version__)

TF version: 2.21.0
TFDV version: 1.21.0


## Download and prepare the data

The raw file encodes `season` and `weathersit` as integers. You will map them to readable strings so TFDV treats them as categorical features with string domains.

You will then cut the data into batches, one folder per batch (a layout TFX's `CsvExampleGen` can read directly):

| Batch | Purpose |
|---|---|
| `2011-Q1` | Baseline for schema inference |
| `2011-04` to `2011-09` | Monthly training batches to monitor |
| `2011-08` | Corrupted by a simulated vendor bug |
| `2011-10-serving` | Serving batch: label columns removed |
| `2012-07` | Held back for a future pipeline run (the vendor bug recurs) |

In [4]:
DATA_DIR = 'data/batches'
ARTIFACT_DIR = 'artifacts'
DB_PATH = 'metadata/monitoring.sqlite'

for d in ['raw', DATA_DIR, f'{ARTIFACT_DIR}/stats', f'{ARTIFACT_DIR}/schemas',
          f'{ARTIFACT_DIR}/anomalies', 'metadata']:
    os.makedirs(d, exist_ok=True)

url = 'https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
if not os.path.exists('raw/hour.csv'):
    zip_path, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(zip_path) as z:
        member = [n for n in z.namelist() if n.endswith('hour.csv')][0]
        with z.open(member) as src, open('raw/hour.csv', 'wb') as dst:
            dst.write(src.read())

print(pd.read_csv('raw/hour.csv', nrows=3))

   instant      dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
0        1  2011-01-01       1   0     1   0        0        6           0   
1        2  2011-01-01       1   0     1   1        0        6           0   
2        3  2011-01-01       1   0     1   2        0        6           0   

   weathersit  temp   atemp   hum  windspeed  casual  registered  cnt  
0           1  0.24  0.2879  0.81          0       3          13   16  
1           1  0.22  0.2727  0.80          0       8          32   40  
2           1  0.22  0.2727  0.80          0       5          27   32  


In [5]:
SEASONS = {1: 'winter', 2: 'spring', 3: 'summer', 4: 'fall'}
WEATHER = {1: 'clear', 2: 'mist', 3: 'light_precip', 4: 'heavy_precip'}
LABELS = ['cnt', 'casual', 'registered']

hour = pd.read_csv('raw/hour.csv', parse_dates=['dteday'])
hour['season'] = hour['season'].map(SEASONS)
hour['weathersit'] = hour['weathersit'].map(WEATHER)
hour['period'] = hour['dteday'].dt.strftime('%Y-%m')
hour = hour.drop(columns=['instant', 'dteday'])


def write_batch(df, name):
    folder = os.path.join(DATA_DIR, name)
    os.makedirs(folder, exist_ok=True)
    path = os.path.abspath(os.path.join(folder, 'data.csv'))
    df.drop(columns=['period']).to_csv(path, index=False)
    return path


def corrupt_vendor_feed(df, seed=42):
    """Simulate a vendor bug: temperature sent in Celsius, some windspeed values dropped."""
    bad = df.copy()
    bad['temp'] = (bad['temp'] * 41).round(2)   # normalized temp = Celsius / 41
    bad.loc[bad.sample(frac=0.05, random_state=seed).index, 'windspeed'] = np.nan
    return bad


MONTHS = ['2011-04', '2011-05', '2011-06', '2011-07', '2011-08', '2011-09']

batch_paths = {'2011-Q1': write_batch(hour[hour['period'].isin(['2011-01', '2011-02', '2011-03'])], '2011-Q1')}
for m in MONTHS:
    df = hour[hour['period'] == m]
    batch_paths[m] = write_batch(corrupt_vendor_feed(df) if m == '2011-08' else df, m)

serving = hour[hour['period'] == '2011-10'].drop(columns=LABELS)
batch_paths['2011-10-serving'] = write_batch(serving, '2011-10-serving')

write_batch(corrupt_vendor_feed(hour[hour['period'] == '2012-07']), '2012-07')   # held back

for name, path in batch_paths.items():
    print(f'{name:<16} {len(pd.read_csv(path)):>5} rows  {path}')

2011-Q1           2067 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitoring/data/batches/2011-Q1/data.csv
2011-04            719 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitoring/data/batches/2011-04/data.csv
2011-05            744 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitoring/data/batches/2011-05/data.csv
2011-06            720 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitoring/data/batches/2011-06/data.csv
2011-07            744 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitoring/data/batches/2011-07/data.csv
2011-08            731 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitoring/data/batches/2011-08/data.csv
2011-09            717 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitoring/data/batches/2011-09/data.csv
2011-10-serving    743 rows  /Users/nainicadasari/MLOps-course/Labs/MLMD_Labs/Lab1_Data_Monitorin

## Set up a persistent metadata store

The metadata store is a SQLite file, so the audit trail outlives the notebook and can be queried later.

In [6]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)   # start clean each time you run the notebook top to bottom

connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.sqlite.filename_uri = DB_PATH
connection_config.sqlite.connection_mode = metadata_store_pb2.SqliteMetadataSourceConfig.READWRITE_OPENCREATE
store = metadata_store.MetadataStore(connection_config)

## Register types

| Kind | Name | Properties | Role in this lab |
|---|---|---|---|
| ArtifactType | `DataBatch` | period, environment, num_rows | A CSV batch to validate |
| ArtifactType | `ExampleStatistics` | period | TFDV statistics for a batch |
| ArtifactType | `Schema` | version, note, sha256 | Every schema version, inferred or curated |
| ArtifactType | `ExampleAnomalies` | period, num_anomalies, decision | One validation result |
| ExecutionType | `GenerateStatistics` | state | |
| ExecutionType | `InferSchema` | state | |
| ExecutionType | `CurateSchema` | state, reason | Every human or policy-driven schema change |
| ExecutionType | `ValidateBatch` | state, environment, drift_check | |
| ContextType | `MonitoringCycle` | period | Groups everything done for one month |

The `sha256` on `Schema` lets you recognize a schema file outside this store, for example after a pipeline copies it.

In [7]:
S, I, D = metadata_store_pb2.STRING, metadata_store_pb2.INT, metadata_store_pb2.DOUBLE

ARTIFACT_TYPES = {
    'DataBatch': {'period': S, 'environment': S, 'num_rows': I},
    'ExampleStatistics': {'period': S},
    'Schema': {'version': I, 'note': S, 'sha256': S},
    'ExampleAnomalies': {'period': S, 'num_anomalies': I, 'decision': S},
}
EXECUTION_TYPES = {
    'GenerateStatistics': {'state': S},
    'InferSchema': {'state': S},
    'CurateSchema': {'state': S, 'reason': S},
    'ValidateBatch': {'state': S, 'environment': S, 'drift_check': I},
}

TYPE_IDS = {}
for name, props in ARTIFACT_TYPES.items():
    t = metadata_store_pb2.ArtifactType(name=name)
    for k, v in props.items():
        t.properties[k] = v
    TYPE_IDS[name] = store.put_artifact_type(t)

for name, props in EXECUTION_TYPES.items():
    t = metadata_store_pb2.ExecutionType(name=name)
    for k, v in props.items():
        t.properties[k] = v
    TYPE_IDS[name] = store.put_execution_type(t)

cycle_type = metadata_store_pb2.ContextType(name='MonitoringCycle')
cycle_type.properties['period'] = S
TYPE_IDS['MonitoringCycle'] = store.put_context_type(cycle_type)

TYPE_IDS

{'DataBatch': 10,
 'ExampleStatistics': 11,
 'Schema': 12,
 'ExampleAnomalies': 13,
 'GenerateStatistics': 14,
 'InferSchema': 15,
 'CurateSchema': 16,
 'ValidateBatch': 17,
 'MonitoringCycle': 18}

## Recording helpers

Every step is recorded the same way: create the execution as `RUNNING`, run the work, register the output artifacts, put the input and output events, mark the execution `COMPLETED`, and link everything to the month's context. The helpers below wrap those MLMD API calls. Two details:

- Each event records the artifact's **role** in `event.path` (`statistics`, `schema`, `previous_statistics`, `trigger`). Without roles, a validation with two `ExampleStatistics` inputs (current and previous month) can't be told apart later.
- Outputs are attributed to the month's `MonitoringCycle` context, and executions are associated with it.

In [8]:
def _set(value, v):
    if isinstance(v, (bool, int, np.integer)):
        value.int_value = int(v)
    elif isinstance(v, float):
        value.double_value = v
    else:
        value.string_value = str(v)


def sha256(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


def new_artifact(type_name, uri, properties):
    a = metadata_store_pb2.Artifact(type_id=TYPE_IDS[type_name], uri=uri)
    for k, v in properties.items():
        _set(a.properties[k], v)
    a.id = store.put_artifacts([a])[0]
    return a


def new_cycle(name, period):
    c = metadata_store_pb2.Context(type_id=TYPE_IDS['MonitoringCycle'], name=name)
    c.properties['period'].string_value = period
    c.id = store.put_contexts([c])[0]
    return c


def start_execution(type_name, properties=None):
    e = metadata_store_pb2.Execution(type_id=TYPE_IDS[type_name])
    e.properties['state'].string_value = 'RUNNING'
    for k, v in (properties or {}).items():
        _set(e.properties[k], v)
    e.id = store.put_executions([e])[0]
    return e


def finish_execution(execution, inputs, outputs, context, state='COMPLETED'):
    """inputs/outputs: {role: artifact}. Records events, final state, attributions, association."""
    events = []
    for event_type, artifacts in [(metadata_store_pb2.Event.INPUT, inputs),
                                  (metadata_store_pb2.Event.OUTPUT, outputs)]:
        for role, artifact in artifacts.items():
            ev = metadata_store_pb2.Event(artifact_id=artifact.id, execution_id=execution.id, type=event_type)
            ev.path.steps.add().key = role
            events.append(ev)
    store.put_events(events)

    execution.properties['state'].string_value = state
    store.put_executions([execution])

    attributions = [metadata_store_pb2.Attribution(artifact_id=a.id, context_id=context.id)
                    for a in outputs.values()]
    association = metadata_store_pb2.Association(execution_id=execution.id, context_id=context.id)
    store.put_attributions_and_associations(attributions, [association])

## Pipeline steps

Each step does real TFDV work and records itself with the helpers above.

In [9]:
def register_batch(period, path, environment, context):
    batch = new_artifact('DataBatch', path, {'period': period, 'environment': environment,
                                             'num_rows': len(pd.read_csv(path))})
    store.put_attributions_and_associations(
        [metadata_store_pb2.Attribution(artifact_id=batch.id, context_id=context.id)], [])
    return batch


def generate_statistics(batch, context):
    exe = start_execution('GenerateStatistics')
    stats = tfdv.generate_statistics_from_csv(data_location=batch.uri)

    period = batch.properties['period'].string_value
    env = batch.properties['environment'].string_value.lower()
    path = os.path.abspath(f'{ARTIFACT_DIR}/stats/{period}_{env}.pbtxt')
    tfdv.write_stats_text(stats, path)

    stats_artifact = new_artifact('ExampleStatistics', path, {'period': period})
    finish_execution(exe, {'examples': batch}, {'statistics': stats_artifact}, context)
    return stats_artifact, stats


def save_schema(schema, version, note):
    path = os.path.abspath(f'{ARTIFACT_DIR}/schemas/schema_v{version}.pbtxt')
    tfdv.write_schema_text(schema, path)
    return new_artifact('Schema', path, {'version': version, 'note': note, 'sha256': sha256(path)})

## Baseline: infer schema v0 and curate schema v1

The inferred schema and the curated schema are recorded as **separate versions**. The curated one is produced by a `CurateSchema` execution whose input is the inferred one, so the store records that one was derived from the other.

In [10]:
baseline_ctx = new_cycle('cycle-2011-Q1-baseline', '2011-Q1')
baseline_batch = register_batch('2011-Q1', batch_paths['2011-Q1'], 'TRAINING', baseline_ctx)
baseline_stats_art, baseline_stats = generate_statistics(baseline_batch, baseline_ctx)

exe = start_execution('InferSchema')
inferred = tfdv.infer_schema(statistics=baseline_stats)
inferred_art = save_schema(inferred, 0, 'inferred from 2011-Q1 baseline')
finish_execution(exe, {'statistics': baseline_stats_art}, {'schema': inferred_art}, baseline_ctx)

tfdv.display_schema(inferred)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


,Type,Presence,Valency,Domain
Feature name,,,,
'season',STRING,required,,'season'
'yr',INT,required,,-
'mnth',INT,required,,-
'hr',INT,required,,-
'holiday',INT,required,,-
'weekday',INT,required,,-
'workingday',INT,required,,-
'weathersit',STRING,required,,'weathersit'
'temp',FLOAT,required,,-


,Values
Domain,
'season',"'spring', 'winter'"
'weathersit',"'clear', 'heavy_precip', 'light_precip', 'mist'"


Look at the inferred schema. `season` only knows `winter` and `spring`, because the baseline ends in March. `temp`, `hum` and `windspeed` have no range. Inference only knows what it has seen.

Curate it using the dataset's documentation, not just the baseline:

- `hr` is always 0 to 23.
- `temp`, `atemp`, `hum` and `windspeed` are normalized to [0, 1].
- `weathersit` has four documented values. Add all four now, because a rare value like `heavy_precip` is legitimate even if Q1 never saw it.
- `season` is **deliberately left as inferred**. You'll see what happens when summer arrives.
- The label columns (`cnt`, `casual`, `registered`) are excluded from the `SERVING` environment.
- Drift comparators: L-infinity distance on `weathersit` and Jensen-Shannon divergence on `temp` and `hum`.

In [11]:
curated = schema_pb2.Schema()
curated.CopyFrom(inferred)

tfdv.set_domain(curated, 'hr', schema_pb2.IntDomain(name='hr', min=0, max=23))
for f in ['temp', 'atemp', 'hum', 'windspeed']:
    tfdv.set_domain(curated, f, schema_pb2.FloatDomain(name=f, min=0.0, max=1.0))

weather_domain = tfdv.get_domain(curated, 'weathersit')
for value in WEATHER.values():
    if value not in weather_domain.value:
        weather_domain.value.append(value)

curated.default_environment.append('TRAINING')
curated.default_environment.append('SERVING')
for f in LABELS:
    tfdv.get_feature(curated, f).not_in_environment.append('SERVING')

tfdv.get_feature(curated, 'weathersit').drift_comparator.infinity_norm.threshold = 0.15
for f in ['temp', 'hum']:
    tfdv.get_feature(curated, f).drift_comparator.jensen_shannon_divergence.threshold = 0.25

exe = start_execution('CurateSchema', {'reason': 'baseline curation from data dictionary'})
curated_art = save_schema(curated, 1, 'curated: domains, environments, drift comparators')
finish_execution(exe, {'base_schema': inferred_art}, {'schema': curated_art}, baseline_ctx)

tfdv.display_schema(curated)

,Type,Presence,Valency,Domain
Feature name,,,,
'season',STRING,required,,'season'
'yr',INT,required,,-
'mnth',INT,required,,-
'hr',INT,required,,min: 0; max: 23
'holiday',INT,required,,-
'weekday',INT,required,,-
'workingday',INT,required,,-
'weathersit',STRING,required,,'weathersit'
'temp',FLOAT,required,,min: 0.000000; max: 1.000000


,Values
Domain,
'season',"'spring', 'winter'"
'weathersit',"'clear', 'heavy_precip', 'light_precip', 'mist'"


## Decision policy

TFDV tells you *what* is anomalous. Your team still has to decide what to *do*. The policy below sorts every anomalous feature into one of three kinds, using the reason codes in the `Anomalies` proto:

- **drift**: every reason is a `COMPARATOR_*` code. The distribution moved, but the values are valid.
- **new_category**: only `season` has new string values. The calendar moved on, so this is a legitimate change that needs a schema update.
- **quality**: anything else (out-of-range values, missing values, missing columns). The data is broken.

The batch decision is then: any quality issue gives **REJECTED**; otherwise a new category gives **NEEDS_SCHEMA_UPDATE**; otherwise drift gives **ACCEPTED_WITH_DRIFT**; otherwise **ACCEPTED**.

In [12]:
def classify(anomalies):
    kinds = {}
    for feature, info in anomalies.anomaly_info.items():
        reasons = {anomalies_pb2.AnomalyInfo.Type.Name(r.type) for r in info.reason}
        if reasons and all(r.startswith('COMPARATOR') for r in reasons):
            kinds[feature] = 'drift'
        elif feature == 'season' and reasons == {'ENUM_TYPE_UNEXPECTED_STRING_VALUES'}:
            kinds[feature] = 'new_category'
        else:
            kinds[feature] = 'quality'
    return kinds


def decide(kinds):
    found = set(kinds.values())
    if 'quality' in found:
        return 'REJECTED'
    if 'new_category' in found:
        return 'NEEDS_SCHEMA_UPDATE'
    if 'drift' in found:
        return 'ACCEPTED_WITH_DRIFT'
    return 'ACCEPTED'

In [13]:
def validate(stats_art, stats, schema_art, schema, context, environment='TRAINING', previous=None):
    """previous: (stats_artifact, stats) of the last ACCEPTED batch, used for drift detection."""
    exe = start_execution('ValidateBatch', {'environment': environment,
                                            'drift_check': int(previous is not None)})
    anomalies = tfdv.validate_statistics(
        statistics=stats, schema=schema, environment=environment,
        previous_statistics=previous[1] if previous else None)
    decision = decide(classify(anomalies))

    period = stats_art.properties['period'].string_value
    version = schema_art.properties['version'].int_value
    path = os.path.abspath(f'{ARTIFACT_DIR}/anomalies/{period}_{environment.lower()}_v{version}.pbtxt')
    tfdv.write_anomalies_text(anomalies, path)

    anomalies_art = new_artifact('ExampleAnomalies', path, {
        'period': period, 'num_anomalies': len(anomalies.anomaly_info), 'decision': decision})
    inputs = {'statistics': stats_art, 'schema': schema_art}
    if previous:
        inputs['previous_statistics'] = previous[0]
    finish_execution(exe, inputs, {'anomalies': anomalies_art}, context)
    return anomalies_art, anomalies, decision


def evolve_schema(schema_art, schema, trigger_art, batch, context):
    """Add new season values. The anomaly report that forced the change is recorded as an input."""
    period = batch.properties['period'].string_value
    exe = start_execution('CurateSchema', {'reason': f'new season values observed in {period}'})

    new_schema = schema_pb2.Schema()
    new_schema.CopyFrom(schema)
    domain = tfdv.get_domain(new_schema, 'season')
    seen = pd.read_csv(batch.uri, usecols=['season'])['season'].dropna().unique()
    added = [s for s in seen if s not in domain.value]
    domain.value.extend(added)

    version = schema_art.properties['version'].int_value + 1
    new_art = save_schema(new_schema, version, f'added season values {added} after {period}')
    finish_execution(exe, {'base_schema': schema_art, 'trigger': trigger_art}, {'schema': new_art}, context)
    return new_art, new_schema


def set_status(batch, decision):
    batch.custom_properties['status'].string_value = decision
    store.put_artifacts([batch])

## Run the monitoring loop

For each month:

1. Register the batch and compute statistics.
2. Validate against the current schema and against the **last accepted** month. A rejected month must never become the drift reference.
3. If the decision is `NEEDS_SCHEMA_UPDATE`, evolve the schema and validate again.
4. Store the final decision on the batch as a custom property.

In [14]:
schema_art, schema = curated_art, curated
previous = (baseline_stats_art, baseline_stats)
results = {}

for period in MONTHS:
    ctx = new_cycle(f'cycle-{period}', period)
    batch = register_batch(period, batch_paths[period], 'TRAINING', ctx)
    stats_art, stats = generate_statistics(batch, ctx)

    anomalies_art, anomalies, decision = validate(stats_art, stats, schema_art, schema, ctx, previous=previous)
    print(f'{period}  schema v{schema_art.properties["version"].int_value}  {decision:<20} {classify(anomalies)}')

    if decision == 'NEEDS_SCHEMA_UPDATE':
        schema_art, schema = evolve_schema(schema_art, schema, anomalies_art, batch, ctx)
        anomalies_art, anomalies, decision = validate(stats_art, stats, schema_art, schema, ctx, previous=previous)
        print(f'{period}  schema v{schema_art.properties["version"].int_value}  {decision:<20} {classify(anomalies)}  (re-validated)')

    set_status(batch, decision)
    results[period] = anomalies
    if decision != 'REJECTED':
        previous = (stats_art, stats)

2011-04  schema v1  ACCEPTED_WITH_DRIFT  {'temp': 'drift'}
2011-05  schema v1  ACCEPTED             {}
2011-06  schema v1  NEEDS_SCHEMA_UPDATE  {'season': 'new_category', 'temp': 'drift'}
2011-06  schema v2  ACCEPTED_WITH_DRIFT  {'temp': 'drift'}  (re-validated)
2011-07  schema v2  ACCEPTED             {}
2011-08  schema v2  REJECTED             {'temp': 'quality', 'windspeed': 'quality'}
2011-09  schema v2  NEEDS_SCHEMA_UPDATE  {'season': 'new_category', 'weathersit': 'drift', 'hum': 'drift', 'temp': 'drift'}
2011-09  schema v3  ACCEPTED_WITH_DRIFT  {'weathersit': 'drift', 'hum': 'drift', 'temp': 'drift'}  (re-validated)


The August batch was rejected. The anomalies below show the features that failed and the reason codes TFDV reported.

In [15]:
tfdv.display_anomalies(results['2011-08'])

,Anomaly short description,Anomaly long description,Anomaly types
Feature name,,,
'temp',Multiple errors,"Unexpectedly high value: 37.72>1(upto six significant digits) The approximate Jensen-Shannon divergence between current and previous is 1 (up to six significant digits), above the threshold 0.25.",FLOAT_TYPE_BIG_FLOAT; COMPARATOR_JENSEN_SHANNON_DIVERGENCE_HIGH
'windspeed',Multiple errors,"The feature has a shape, but it's not always present (if the feature is nested, then it should always be present at each nested level) or its value lengths vary. The feature was present in fewer examples than expected: minimum fraction = 1.000000, actual = 0.949384",INVALID_FEATURE_SHAPE; FEATURE_TYPE_LOW_FRACTION_PRESENT


## Validate a serving batch with environments

The October batch simulates serving traffic: the label columns are missing. You will validate it twice, once in each environment, and record both.

In [16]:
serving_ctx = new_cycle('cycle-2011-10-serving', '2011-10')
serving_batch = register_batch('2011-10-serving', batch_paths['2011-10-serving'], 'SERVING', serving_ctx)
serving_stats_art, serving_stats = generate_statistics(serving_batch, serving_ctx)

for env in ['TRAINING', 'SERVING']:
    _, anomalies, decision = validate(serving_stats_art, serving_stats, schema_art, schema, serving_ctx, environment=env)
    print(f'{env:<9} {decision:<20} {classify(anomalies)}')
    if env == 'SERVING':
        set_status(serving_batch, decision)

TRAINING  REJECTED             {'cnt': 'quality', 'registered': 'quality', 'casual': 'quality'}
SERVING   ACCEPTED             {}


## Audit the store

Everything below uses **only the metadata store**: no Python variables from the loop above. Imagine you joined the team six months later and all you have is `monitoring.sqlite`.

Three graph helpers:

- `producer`: the execution that output an artifact.
- `inputs_of`: an execution's inputs, keyed by role.
- `consumers`: the executions that used an artifact as input.

In [17]:
def producer(artifact_id):
    ids = [e.execution_id for e in store.get_events_by_artifact_ids([artifact_id])
           if e.type == metadata_store_pb2.Event.OUTPUT]
    return store.get_executions_by_id(ids)[0] if ids else None


def inputs_of(execution_id):
    events = [e for e in store.get_events_by_execution_ids([execution_id])
              if e.type == metadata_store_pb2.Event.INPUT]
    artifacts = {a.id: a for a in store.get_artifacts_by_id([e.artifact_id for e in events])}
    return {e.path.steps[0].key: artifacts[e.artifact_id] for e in events}


def consumers(artifact_id):
    ids = [e.execution_id for e in store.get_events_by_artifact_ids([artifact_id])
           if e.type == metadata_store_pb2.Event.INPUT]
    return store.get_executions_by_id(ids)


EXEC_TYPE_NAMES = {t.id: t.name for t in store.get_execution_types()}

### Audit 1: the monitoring dashboard

One row per validation: which batch, which schema version, which environment, what was found, what was decided. The table is built by walking anomalies → `ValidateBatch` → statistics → `GenerateStatistics` → batch.

In [18]:
rows = []
for anomalies_art in sorted(store.get_artifacts_by_type('ExampleAnomalies'), key=lambda a: a.id):
    validation = producer(anomalies_art.id)
    ins = inputs_of(validation.id)
    batch = inputs_of(producer(ins['statistics'].id).id)['examples']
    found = tfdv.load_anomalies_text(anomalies_art.uri)
    rows.append({
        'batch': batch.properties['period'].string_value,
        'environment': validation.properties['environment'].string_value,
        'schema_v': ins['schema'].properties['version'].int_value,
        'drift_vs': ins['previous_statistics'].properties['period'].string_value if 'previous_statistics' in ins else '-',
        'anomalous_features': ', '.join(sorted(found.anomaly_info)) or '-',
        'decision': anomalies_art.properties['decision'].string_value,
        'final_batch_status': batch.custom_properties['status'].string_value if 'status' in batch.custom_properties else '-',
    })

dashboard = pd.DataFrame(rows)
dashboard

,batch,environment,schema_v,drift_vs,anomalous_features,decision,final_batch_status
0,2011-04,TRAINING,1,2011-Q1,temp,ACCEPTED_WITH_DRIFT,ACCEPTED_WITH_DRIFT
1,2011-05,TRAINING,1,2011-04,-,ACCEPTED,ACCEPTED
2,2011-06,TRAINING,1,2011-05,"season, temp",NEEDS_SCHEMA_UPDATE,ACCEPTED_WITH_DRIFT
3,2011-06,TRAINING,2,2011-05,temp,ACCEPTED_WITH_DRIFT,ACCEPTED_WITH_DRIFT
4,2011-07,TRAINING,2,2011-06,-,ACCEPTED,ACCEPTED
5,2011-08,TRAINING,2,2011-07,"temp, windspeed",REJECTED,REJECTED
6,2011-09,TRAINING,2,2011-07,"hum, season, temp, weathersit",NEEDS_SCHEMA_UPDATE,ACCEPTED_WITH_DRIFT
7,2011-09,TRAINING,3,2011-07,"hum, temp, weathersit",ACCEPTED_WITH_DRIFT,ACCEPTED_WITH_DRIFT
8,2011-10-serving,TRAINING,3,-,"casual, cnt, registered",REJECTED,ACCEPTED
9,2011-10-serving,SERVING,3,-,-,ACCEPTED,ACCEPTED


### Audit 2: why does each schema version exist?

For every `Schema`, find the execution that produced it, its reason, and (for evolutions) the anomaly report that triggered it.

In [19]:
for s in sorted(store.get_artifacts_by_type('Schema'), key=lambda a: a.properties['version'].int_value):
    exe = producer(s.id)
    ins = inputs_of(exe.id)
    line = f"v{s.properties['version'].int_value}: {EXEC_TYPE_NAMES[exe.type_id]}"
    if 'reason' in exe.properties:
        line += f" - {exe.properties['reason'].string_value}"
    if 'base_schema' in ins:
        line += f" (from v{ins['base_schema'].properties['version'].int_value})"
    if 'trigger' in ins:
        trig = ins['trigger']
        line += (f"\n    triggered by anomalies on {trig.properties['period'].string_value}: "
                 f"{', '.join(tfdv.load_anomalies_text(trig.uri).anomaly_info)}")
    print(line)

v0: InferSchema
v1: CurateSchema - baseline curation from data dictionary (from v0)
v2: CurateSchema - new season values observed in 2011-06 (from v1)
    triggered by anomalies on 2011-06: season, temp
v3: CurateSchema - new season values observed in 2011-09 (from v2)
    triggered by anomalies on 2011-09: season, weathersit, hum, temp


### Audit 3: impact of each schema version

For every schema version, list the batches validated against it. If a schema version were found to be wrong, these are the batches whose decisions you would revisit.

In [20]:
for s in sorted(store.get_artifacts_by_type('Schema'), key=lambda a: a.properties['version'].int_value):
    validations = [x for x in consumers(s.id) if EXEC_TYPE_NAMES[x.type_id] == 'ValidateBatch']
    used_by = []
    for v in validations:
        stats = inputs_of(v.id)['statistics']
        used_by.append(f"{stats.properties['period'].string_value}/{v.properties['environment'].string_value}")
    print(f"v{s.properties['version'].int_value}: {', '.join(used_by) or 'never used for validation'}")

v0: never used for validation
v1: 2011-04/TRAINING, 2011-05/TRAINING, 2011-06/TRAINING
v2: 2011-06/TRAINING, 2011-07/TRAINING, 2011-08/TRAINING, 2011-09/TRAINING
v3: 2011-09/TRAINING, 2011-10-serving/TRAINING, 2011-10-serving/SERVING


## Summary

This lab built a data-quality monitoring loop on monthly batches of the Bike Sharing data, with every step recorded in an ML Metadata store.

- **Rejecting broken data:** the August batch carried a simulated vendor bug (temperature in Celsius, missing windspeed values). The policy rejected it instead of changing the schema to fit bad data, and it was excluded as the drift reference for September.
- **Evolving the schema for legitimate change:** new seasons in June and September produced schema versions v2 and v3. Each version is linked in the store to the anomaly report that triggered it.
- **Environments:** the serving batch passes under `SERVING`, where labels are not expected, and fails under `TRAINING`, where they are.
- **Auditability:** the dashboard, schema history and schema-impact queries were all built from the metadata store alone. Any decision can be traced back to the data, schema version and anomalies behind it, without rerunning the pipeline.

In [21]:
# Validation checks: every assertion must pass
from collections import Counter

# 1. Every execution recorded in the store finished
states = Counter(e.properties['state'].string_value for e in store.get_executions())
assert set(states) == {'COMPLETED'}, states

# 2. Ground truth from the raw batch files: only August has Celsius temperatures or missing windspeed
corrupted = [p for p, path in batch_paths.items()
             if pd.read_csv(path)['temp'].max() > 1 or pd.read_csv(path)['windspeed'].isna().any()]
assert corrupted == ['2011-08'], corrupted

# 3. Final batch decisions
final = dict(zip(dashboard['batch'], dashboard['final_batch_status']))
assert final['2011-08'] == 'REJECTED', final
assert final['2011-06'] != 'REJECTED' and final['2011-09'] != 'REJECTED', final
assert final['2011-10-serving'] == 'ACCEPTED', final
assert all(final[m] != 'REJECTED' for m in ['2011-04', '2011-05', '2011-07']), final

# 4. Schema versions, and the final season domain
versions = sorted(s.properties['version'].int_value for s in store.get_artifacts_by_type('Schema'))
assert versions == [0, 1, 2, 3], versions
assert set(tfdv.get_domain(schema, 'season').value) == {'winter', 'spring', 'summer', 'fall'}

# 5. The rejected month was never used as a drift reference
assert '2011-08' not in set(dashboard['drift_vs']), dashboard['drift_vs'].tolist()

print('All validation checks passed:', dict(states))

All validation checks passed: {'COMPLETED': 22}
